# Shotwise worker on Kaggle
Enable a GPU session, add a secret `RCLONE_CONF` with your rclone config (a Google Drive remote named `gdrive`), then run the cells in order. The worker drains the queue once and exits, so a session never idles.

In [ ]:
!git clone -q https://github.com/<you>/shotwise.git
!git clone -q https://github.com/comfyanonymous/ComfyUI.git
%pip install -q -r ComfyUI/requirements.txt websocket-client
%pip install -q -e shotwise

In [ ]:
# model weights: Wan 2.2 TI2V-5B (fp8), Qwen-Image-Edit, LatentSync. Edit scripts/get_models.sh to add or remove.
!bash shotwise/scripts/get_models.sh ComfyUI/models

In [ ]:
import subprocess, time
comfy = subprocess.Popen(['python', 'ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--lowvram'])
time.sleep(40)

In [ ]:
!curl -s https://rclone.org/install.sh | bash >/dev/null
from kaggle_secrets import UserSecretsClient
import os; os.makedirs('/root/.config/rclone', exist_ok=True)
open('/root/.config/rclone/rclone.conf', 'w').write(UserSecretsClient().get_secret('RCLONE_CONF'))
!rclone copy gdrive:shotwise/queue /kaggle/working/queue
!rclone copy gdrive:shotwise/workflows /kaggle/working/workflows

In [ ]:
!python -m shotwise.workers.comfy_worker --queue /kaggle/working/queue --workflows /kaggle/working/workflows --once
!rclone copy /kaggle/working/queue/done gdrive:shotwise/queue/done
!rclone sync /kaggle/working/queue/pending gdrive:shotwise/queue/pending